In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
# Lendo os arquivos csv
orders = pd.read_csv('/content/sample_data/orders.csv', sep=';')
order_details = pd.read_csv('/content/sample_data/order_details.csv', sep=';')
products = pd.read_csv('/content/sample_data/products.csv', sep=';')
categories = pd.read_csv('/content/sample_data/categories.csv', sep=';')
customers = pd.read_csv('/content/sample_data/customers.csv', sep=';')
employees = pd.read_csv('/content/sample_data/employees.csv', sep=';')
employee_territories = pd.read_csv('/content/sample_data/employee_territories.csv', sep=';')
territories = pd.read_csv('/content/sample_data/territories.csv', sep=';')
region = pd.read_csv('/content/sample_data/region.csv', sep=';')
us_states = pd.read_csv('/content/sample_data/us_states.csv', sep=';')

print("Todas as tabelas foram carregadas com sucesso!")

Todas as tabelas foram carregadas com sucesso!


In [ ]:
# Unindo tabela de detalhes do pedido com a tabela de pedidos
fact_sales = pd.merge(order_details, orders, on='order_id')

# Verificação do tipo de dado (float)
fact_sales['unit_price'] = fact_sales['unit_price'].astype(float)
fact_sales['quantity'] = fact_sales['quantity'].astype(float)
fact_sales['discount'] = fact_sales['discount'].astype(float)

# Cálculo com round para evitar dízimas infinitas
fact_sales['net_revenue'] = (fact_sales['unit_price'] * fact_sales['quantity'] * (1 - fact_sales['discount'])).round(2)

# Validação: Qual o faturamento total da Northwind?
print(f"Faturamento Total: ${fact_sales['net_revenue'].sum():,.2f}")

# Tratamento de Datas
fact_sales['order_date'] = pd.to_datetime(fact_sales['order_date'])

# Input da tabela fato vendas
fact_sales.head()

Faturamento Total: $1,265,793.02


,order_id,product_id,unit_price,quantity,discount,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country,net_revenue
0,10248,11,14.0,12.0,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NaN,51100,France,168.0
1,10248,42,9.8,10.0,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NaN,51100,France,98.0
2,10248,72,34.8,5.0,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NaN,51100,France,174.0
3,10249,14,18.6,9.0,0.0,TOMSP,6,1996-07-05,1996-08-16,1996-07-10,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,NaN,44087,Germany,167.4
4,10249,51,42.4,40.0,0.0,TOMSP,6,1996-07-05,1996-08-16,1996-07-10,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,NaN,44087,Germany,1696.0


In [ ]:
# Criando tabela dimensão de produtos (Produto + Categoria)

dim_products = pd.merge(products, categories, on='category_id', how='left')

# Criando tabela dimensão de território (Território + Região)
dim_territories = pd.merge(territories, region, on='region_id', how='left')

# Criando tabela dimensão de clientes
dim_customers = customers.copy()

print("Tabelas dimensão prontas para exportação.")

Tabelas dimensão prontas para exportação.


In [ ]:
# Data da última compra geral na empresa para usar como referência
last_purchase = fact_sales['order_date'].max()

# Data da última compra de cada cliente
churn_df = fact_sales.groupby('customer_id')['order_date'].max().reset_index()
churn_df['days_since_last_purchase'] = (last_purchase - churn_df['order_date']).dt.days

# Clientes que estão sem compras há mais de 90 dias (regra criada de Churn)
churn_df['status_churn'] = np.where(churn_df['days_since_last_purchase'] > 90, 'Churn', 'Ativo')

churn_df.head()

,customer_id,order_date,days_since_last_purchase,status_churn
0,ALFKI,1998-04-09,27,Ativo
1,ANATR,1998-03-04,63,Ativo
2,ANTON,1998-01-28,98,Churn
3,AROUT,1998-04-10,26,Ativo
4,BERGS,1998-03-04,63,Ativo


In [ ]:
fact_sales.to_csv('fato_vendas.csv', index=False, sep=';')
dim_products.to_csv('dim_produtos.csv', index=False, sep=';')
dim_customers.to_csv('dim_clientes.csv', index=False, sep=';')
dim_territories.to_csv('dim_territorios.csv', index=False, sep=';')
churn_df.to_csv('fato_analise_churn.csv', index=False, sep=';')

print("Arquivos csv tratados e já prontos")

Arquivos csv tratados e já prontos


In [ ]:
!pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.7 MB/s eta 0:00:00


In [ ]:
# Exportando as tabelas no formato Parquet

fact_sales.to_parquet('fact_sales.parquet', index=False)
dim_products.to_parquet('dim_products.parquet', index=False)
dim_customers.to_parquet('dim_customers.parquet', index=False)
dim_territories.to_parquet('dim_territories.parquet', index=False)
churn_df.to_parquet('fact_churn_analysis.parquet', index=False)

print("Arquivos convertidos para Parquet!")

Arquivos convertidos para Parquet!


In [ ]:
df = pd.read_parquet('fact_sales.parquet')
df.head()

,order_id,product_id,unit_price,quantity,discount,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country,net_revenue
0,10248,11,14.0,12.0,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,None,51100,France,168.0
1,10248,42,9.8,10.0,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,None,51100,France,98.0
2,10248,72,34.8,5.0,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,None,51100,France,174.0
3,10249,14,18.6,9.0,0.0,TOMSP,6,1996-07-05,1996-08-16,1996-07-10,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,None,44087,Germany,167.4
4,10249,51,42.4,40.0,0.0,TOMSP,6,1996-07-05,1996-08-16,1996-07-10,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,None,44087,Germany,1696.0


In [ ]:
# Usando o csv "us_states" como base para garantir que todos os estados dos EUA existam
dim_states = us_states.copy()
dim_states['country'] = 'USA'

# Exportar para Parquet
dim_states.to_parquet('dim_states.parquet', index=False)

print("Tabela dimensão de estados criada com sucesso!")


Tabela dimensão de estados criada com sucesso!


In [ ]:
fact_sales_completed = pd.merge(
  fact_sales,
  us_states[['state_abbr', 'state_name', 'state_region']],
  left_on='ship_region',
  right_on='state_abbr',
  how='left')

fact_sales_completed.to_parquet('fact_sales_completed.parquet', index=False)

print("Tabela fato vendas completa criada com sucesso!")

Tabela fato vendas completa criada com sucesso!


In [ ]:
# Realizando o join de employees
dim_employees = pd.merge(
  employees,
  employee_territories,
  on='employee_id',
  how='left')

# Criando o Nome Completo de cada employee
dim_employees['full_name'] = dim_employees['first_name'] + ' ' + dim_employees['last_name']

# Exportação para formato Parquet
dim_employees.to_parquet('dim_employees.parquet', index=False)

print(f"Join concluído. A nova dimensão de employees possui {len(dim_employees)} registros.")

dim_employees[['employee_id', 'full_name', 'territory_id']].head()

Join concluído. A nova dimensão de employees possui 49 registros.


,employee_id,full_name,territory_id
0,1,Nancy Davolio,6897
1,1,Nancy Davolio,19713
2,2,Andrew Fuller,1581
3,2,Andrew Fuller,1730
4,2,Andrew Fuller,1833


In [ ]:
from google.colab import drive
import os

# Monta o drive na pasta /content/drive
drive.mount('/content/drive')

# Definindo o caminho da pasta onde os dados da Northwind serão salvos no Google Drive
folder_path = '/content/drive/My Drive/Northwind_BI'

if not os.path.exists(folder_path):
  os.makedirs(folder_path)
  print(f"Pasta criada em: {folder_path}")
else:
  print(f"Pasta já existente em: {folder_path}")

Mounted at /content/drive
Pasta criada em: /content/drive/My Drive/Northwind_BI


In [ ]:
df_sales_completed = pd.read_parquet('fact_sales_completed.parquet')
# Salvando como CSV
df_sales_completed.to_csv('fact_sales_completed.csv', index=False, sep=';', encoding='utf-8-sig')
files.download('fact_sales_completed.csv')

df_products = pd.read_parquet('dim_products.parquet')
# Salvando como CSV
df_products.to_csv('dim_products.csv', index=False, sep=';', encoding='utf-8-sig')
files.download('dim_products.csv')

df_customers = pd.read_parquet('dim_customers.parquet')
# Salvando como CSV
df_customers.to_csv('dim_customers.csv', index=False, sep=';', encoding='utf-8-sig')
files.download('dim_customers.csv')

df_territories = pd.read_parquet('dim_territories.parquet')
# Salvando como CSV
df_territories.to_csv('dim_territories.csv', index=False, sep=';', encoding='utf-8-sig')
files.download('dim_territories.csv')

df_churn_analysis = pd.read_parquet('fact_churn_analysis.parquet')
# Salvando como CSV
df_churn_analysis.to_csv('fact_churn_analysis.csv', index=False, sep=';', encoding='utf-8-sig')
files.download('fact_churn_analysis.csv')

df_employees = pd.read_parquet('dim_employees.parquet')
# Salvando como CSV
df_employees.to_csv('dim_employees.csv', index=False, sep=';', encoding='utf-8-sig')
files.download('dim_employees.csv')

df_states = pd.read_parquet('dim_states.parquet')
# Salvando como CSV
df_states.to_csv('dim_states.csv', index=False, sep=';', encoding='utf-8-sig')
files.download('dim_states.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Dicionário com o nome do arquivo e o DataFrame correspondente
dataframes = {
  'dim_products': df_products,
  'dim_customers': df_customers,
  'dim_territories': df_territories,
  'fact_sales_completed': df_sales_completed,
  'dim_states': df_states,
  'dim employees': df_employees,
  'fact_churn_analysis': df_churn_analysis}

print(f"Arquivos parquet salvos com sucesso no Google Drive!")

Arquivos parquet salvos com sucesso no Google Drive!
